In [0]:
# Databricks notebook source

# COMMAND ----------
# MAGIC %md
# MAGIC # 🥉 BRONZE — Ingestão Raw dos Dados F1
# MAGIC
# MAGIC **Objetivo:** Ler o CSV bruto exatamente como chegou e salvar como Delta Table.
# MAGIC Nenhuma transformação aqui — apenas metadados de rastreabilidade.
# MAGIC
# MAGIC **Antes de rodar:** Suba o CSV pelo menu *Data > Add Data > Upload File*
# MAGIC e anote o caminho exibido (algo como `/FileStore/tables/resultados_f1_com_sprint.csv`).

# COMMAND ----------

# ── Parâmetros ──────────────────────────────────────────────────────────────
CAMINHO_CSV   = "/Workspace/Users/vinicius20204@hotmail.com.br/PROJE-F1-DADOS/resultados_f1_2025_2026.csv"  # ajuste se necessário
DATABASE      = "f1_db"
TABELA_BRONZE = "bronze_resultados_f1"

# COMMAND ----------

# Cria banco de dados se não existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE}")
spark.sql(f"USE {DATABASE}")
print(f"✅ Banco '{DATABASE}' pronto.")

# COMMAND ----------

# Lê o CSV com schema inferido (mantém tudo como string por segurança)
df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")   # bronze = strings puras
    .option("encoding", "UTF-8")
    .csv(CAMINHO_CSV)
)

print(f"📦 Registros lidos: {df_raw.count()}")
print(f"📋 Colunas: {df_raw.columns}")
display(df_raw.limit(10))

# COMMAND ----------

from pyspark.sql import functions as F

# Adiciona colunas de rastreabilidade (obrigatório na camada Bronze)
df_bronze = (
    df_raw
    .withColumn("_source_file",    F.lit(CAMINHO_CSV))
    .withColumn("_ingestion_ts",   F.current_timestamp())
    .withColumn("_ingestion_date", F.current_date())
)

# COMMAND ----------

# Grava como Delta Table (overwrite para reprocessamento seguro)
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.{TABELA_BRONZE}")
)

print(f"✅ Tabela '{DATABASE}.{TABELA_BRONZE}' criada!")
print(f"Total de linhas: {df_bronze.count()}")

# COMMAND ----------

# Validação rápida
spark.sql(f"""
    SELECT tipo, COUNT(*) AS qtd
    FROM {DATABASE}.{TABELA_BRONZE}
    GROUP BY tipo
    ORDER BY tipo
""").show()

✅ Banco 'f1_db' pronto.
📦 Registros lidos: 819
📋 Colunas: ['ano', 'tipo', 'rodada', 'corrida', 'circuito', 'pais', 'data', 'piloto', 'codigo_piloto', 'construtor', 'numero_carro', 'grid', 'posicao', 'pontos', 'voltas', 'status']


ano,tipo,rodada,corrida,circuito,pais,data,piloto,codigo_piloto,construtor,numero_carro,grid,posicao,pontos,voltas,status
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Lando Norris,NOR,McLaren,4,1,1,25.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Max Verstappen,VER,Red Bull,1,3,2,18.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,George Russell,RUS,Mercedes,63,4,3,15.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Andrea Kimi Antonelli,ANT,Mercedes,12,16,4,12.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Alexander Albon,ALB,Williams,23,6,5,10.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Lance Stroll,STR,Aston Martin,18,13,6,8.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Nico Hülkenberg,HUL,Sauber,27,17,7,6.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Charles Leclerc,LEC,Ferrari,16,7,8,4.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Oscar Piastri,PIA,McLaren,81,2,9,2.0,57,Finished
2025,corrida,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Australia,2025-03-16,Lewis Hamilton,HAM,Ferrari,44,8,10,1.0,57,Finished


✅ Tabela 'f1_db.bronze_resultados_f1' criada!
Total de linhas: 819
+-------+---+
|   tipo|qtd|
+-------+---+
|corrida|633|
| sprint|186|
+-------+---+

